# Training the PINN model

## 0. Prerrequesites

In [1]:
import sys
sys.path.append("/scratchsan/observatorio/juagudeloo/Tesis_maestria_OAN/")

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import astropy.units as u
import matplotlib.pyplot as plt

from utils.muram_data import MhdData, StokesData
from utils.normalizer import MhdNormalizer, StokesNormalizer
from models.pinn_mscnn_model import PhysicsInformedMSCNN
from utils.physics_utils import ApproxInversions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Initialization

### 1.1 Charging data

In [2]:
# Data paths (adjust to your system)
data_path = Path("/scratchsan/observatorio/juagudeloo/data/")
step = 112  # Choose a simulation step

# Load MHD data and remap to optical depth
mhd = MhdData(
    data_path=data_path / "muram-simulation",
    nx=480, ny=480, nz=256
)
mhd.load_step(step=step, z_max=250)
mhd.load_opacity_table(kappa_path=data_path / "csv/kappa.0.dat")
mhd.compute_optical_depth(dz=10*u.km)

# Create a uniform log(tau) grid for remapping
new_logtau = np.arange(-2.0, 0.1, 0.1)
mhd.remap_to_optical_depth(new_logtau, quantities=["T", "Vz", "Bz"])

print(f"MHD data loaded: {mhd.od_data['Bz'].shape}")
print(f"Log(tau) values: {new_logtau}")

# Load Stokes data
stokes = StokesData(
    data_dir=data_path / "muram-simulation/",
    step=step,
    wavelength_range=(6300.5, 6303.5),
    wavelength_step=0.01
)
stokes.load_stokes()
stokes.continuum_normalization(cont_indices=[0, 1, 2, 3])
stokes.load_hinode_lsf(data_path / "hinode-MODEST/PSFs/hinode_sp.spline.psf")
stokes.apply_spectral_convolution()
stokes.resample_to_hinode()

print(f"Stokes data loaded: {stokes.data['I'].shape}")
print(f"Stokes wavelength range: {stokes.wl.min():.2f} - {stokes.wl.max():.2f} Å")

Loading step 112000


  Trimmed z axis to first 250 layers (0..249)
  Reference layer (T ~ 5780 K) at z-index 186
  Done.
Opacity interpolator loaded from /scratchsan/observatorio/juagudeloo/data/csv/kappa.0.dat
  T range: [3.320, 5.300] (log10 K)
  P range: [-2.000, 8.000] (log10 dyne/cm²)
Computing optical depth...
  Optical depth computed.
Remapping to optical depth coordinates (21 levels)...
  Processing T
  Processing Vz
  Processing Bz
  Remapping complete.
MHD data loaded: (480, 480, 21)
Log(tau) values: [-2.00000000e+00 -1.90000000e+00 -1.80000000e+00 -1.70000000e+00
 -1.60000000e+00 -1.50000000e+00 -1.40000000e+00 -1.30000000e+00
 -1.20000000e+00 -1.10000000e+00 -1.00000000e+00 -9.00000000e-01
 -8.00000000e-01 -7.00000000e-01 -6.00000000e-01 -5.00000000e-01
 -4.00000000e-01 -3.00000000e-01 -2.00000000e-01 -1.00000000e-01
  1.77635684e-15]
Loading Stokes data from /scratchsan/observatorio/juagudeloo/data/muram-simulation/stokes_112000.npy
  I shape: (480, 480, 300)
  Q shape: (480, 480, 300)
  U sha

### 1.2 Model instantiation

In [3]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate the Physics-Informed MSCNN model
model = PhysicsInformedMSCNN(
    scales=[1, 2, 3],                           # Multi-scale coarse-graining
    in_channels=2,                              # I and V Stokes parameters
    c1_filters=16,                              # Filters in first conv block
    c2_filters=32,                              # Filters in second conv block
    kernel_size=5,                              # Convolution kernel size
    pool_size=2,                                # MaxPool kernel size
    n_linear_layers=4,                          # Dense layers for final mapping
    output_features=3*21,                       # 3 parameters × 21 heights = 63 outputs
    input_length=112,                           # Spectral dimension of Stokes profiles
    
    # Physics-informed parameters
    central_wavelength=6301.5*u.Angstrom,      # Fe I 6301.5 Å
    lande_factor=1.67,                          # Landé g-factor
    wl_range=(15, 60),                          # Wavelength indices for WFA/Doppler
    lambda_reg=0.1,                             # Regularization weight
    use_physics='both',                         # Use both WFA and Doppler (can be None, 'wfa', 'doppler', or 'both')

    # Dropout for uncertainty
    dropout_rate=0.2  # 20% dropout
).to(device)

print(f"Model instantiated on {device}")
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Regularization weight (lambda_reg): {model.lambda_reg}")
print(f"Physics mode: {model.use_physics}")

Using device: cuda


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 3. Tensor creation

In [ ]:
# Step 1: Select a patch
y0, x0 = 100, 200
H, W = 4, 4
n_heights = 21

In [ ]:
stokes_patch = {
    'I': normalized_stokes['I'][y0:y0+H, x0:x0+W, :],
    'V': normalized_stokes['V'][y0:y0+H, x0:x0+W, :],
}

# Step 3: Run model on the patch (batched)
print("\nRunning model inference on patch...")
I_p = torch.from_numpy(stokes_patch['I']).float().to(device)
V_p = torch.from_numpy(stokes_patch['V']).float().to(device)
patch_input = torch.stack([I_p, V_p], dim=2).view(H * W, 2, -1)
print(f"Input patched of shape: {patch_input.shape}")


Running model inference on patch...
Input patched of shape: torch.Size([16, 2, 112])


In [ ]:
# Step 7: Build MHD patch cubes for height selection
mhd_patch_od_data = {
    'T': normalized_mhd['T'][y0:y0+H, x0:x0+W, :],
    'Bz': normalized_mhd['Bz'][y0:y0+H, x0:x0+W, :],
    'Vz': normalized_mhd['Vz'][y0:y0+H, x0:x0+W, :],
}

T_p = torch.from_numpy(mhd_patch_od_data['T'])
Bz_p = torch.from_numpy(mhd_patch_od_data['Bz'])
Vz_p = torch.from_numpy(mhd_patch_od_data['Vz'])

patched_targets = torch.stack([T_p, Bz_p, Vz_p], dim=3).reshape(H * W, n_heights*3)
print(f"Patched targets shape: {patched_targets.shape}")

Patched targets shape: torch.Size([16, 63])


In [ ]:
print("Pre-computing physical approximations (WFA B_LOS and Doppler V_LOS) on patch...")

approx_invs = ApproxInversions(
    stokes=stokes.data,
    wavelength=stokes.wl,
    central_wavelength=model.central_wavelength,
    lande_factor=model.lande_factor,
)
blos_approx = approx_invs.compute_blos_wfa(wl_range=model.wl_range).value
vlos_approx = approx_invs.compute_vlos_doppler(wl_range=model.wl_range).value
print(f"  B_LOS approximation shape: {blos_approx.shape}")
print(f"  V_LOS approximation shape: {vlos_approx.shape}")

blos_approx_patch = blos_approx[y0:y0+H, x0:x0+W]
vlos_approx_patch = vlos_approx[y0:y0+H, x0:x0+W]


Pre-computing physical approximations (WFA B_LOS and Doppler V_LOS) on patch...


  B_LOS approximation shape: (480, 480)
  V_LOS approximation shape: (480, 480)


In [ ]:
bz_idx, bz_rrmse_full = model._best_rrmse_index(blos_approx, mhd.od_data['Bz'].value)
vz_idx, vz_rrmse_full = model._best_rrmse_index(vlos_approx, mhd.od_data['Vz'].value)

print(f"Best Bz index: {bz_idx}, RRMSE: {bz_rrmse_full[bz_idx]:.3f}")
print(f"Best Vz index: {vz_idx}, RRMSE: {vz_rrmse_full[vz_idx]:.3f}")

Best Bz index: 8, RRMSE: 1.147
Best Vz index: 20, RRMSE: 6.270


## 4. Training

## 5. Predicting with uncertainty

NameError: name 'stokes_input' is not defined